# Evaluating PITMuS reconstruction correctness

`gen_dataset.py` turns PIT's `mutations.xml` into `original_method` / `mutated_method`.
This notebook asks: **is `mutated_method` really the mutant PIT intended?**

Each section below is one independent check (oracle):

0. **XML alignment** – each row points back to the exact `<mutation>` it came from (`xml_line`).
1. **Count** – every XML mutation produced a dataset row.
2. **Mutation faithfulness** – the reconstructed edit reflects the *correct mutation*, and the
   mutant is still valid Java. This is the main oracle: it gives every row one verdict
   (**PASS / FAIL(reason) / UNVERIFIABLE**) and is built to raise **no false alarms** — a row
   only FAILs on a genuine defect.
3. **Spot check** – eyeball a random sample.

Set `REPO` / `PROJECT` in the first cell, then run top to bottom.


In [13]:
!pip install javalang

In [20]:
import re, csv, difflib, random
import xml.etree.ElementTree as ET
from pathlib import Path
import javalang

csv.field_size_limit(10 ** 7)  # method bodies can be large

# --- CONFIG: point these at your repo + project ---
REPO    = Path("/Users/nulfat/Documents/Projects/PhD/PITMuS/PITMuS")
PROJECT = "commons-jexl3"                       # any folder under test-projects/
# The reconstruction package lives in PITMuS/pitmus; put PITMuS/ on the path so
# `import pitmus` / pitmus_config resolves without installing.
import sys as _sys; _sys.path.insert(0, str(REPO / "PITMuS"))

DATASET = f"PITMuS_dataset"

proj_dir     = REPO / "test-projects" / PROJECT
meta_path    = proj_dir / DATASET / f"meta-{PROJECT}.csv"
methods_path = proj_dir / DATASET / f"mutated_methods-{PROJECT}.csv"
xml_path     = proj_dir / "target" / "pit-reports" / "mutations.xml"

def load_csv(p):
    with open(p, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

meta    = load_csv(meta_path)                       # one row per mutation
methods = {r["index_no"]: r for r in load_csv(methods_path)}  # index_no -> method bodies

print(f"meta path = {meta_path}")
print(f"project = {PROJECT}")
print(f"meta rows = {len(meta)}   method rows = {len(methods)}")


# ---------- shared helper ----------

def line_tokens(s):
    """Tokenize a Java snippet (the tokenizer ignores comments).
    Returns the list of token strings, or None if the snippet won't tokenize."""
    try:
        return [t.value for t in javalang.tokenizer.tokenize(s)]
    except Exception:
        return None


meta path = /Users/nulfat/Documents/Projects/PhD/PITMuS/PITMuS/test-projects/commons-jexl3/PITMuS_dataset/meta-commons-jexl3.csv
project = commons-jexl3
meta rows = 14019   method rows = 14019


## 0. XML alignment oracle
Does each reconstructed row point back to the exact mutation PIT recorded?

`meta.csv` now carries an **`xml_line`** column: the physical line of that row's `<mutation>`
inside `mutations.xml`. So we can check alignment *exactly* (no fuzzy line matching):

for every row, jump to `xml_line` in the XML and confirm its `sourceFile` + `description`
match the row. A mismatch means `xml_line` points at the wrong mutation.

### Why we `html.unescape` the description first
Some characters (`"`, `<`, `>`, `&`) have special meaning in XML, so they can't be written
literally — the file stores them as **escape codes**:

| in the raw XML file | what it really means |
|---|---|
| `&quot;` | `"` |
| `&lt;`   | `<` |
| `&gt;`   | `>` |
| `&amp;`  | `&` |

In [21]:
import os, html

# read the raw XML text once; xml_line in meta points at the physical line of each <mutation>
with open(xml_path, encoding="utf-8", errors="replace") as f:
    xml_lines = f.readlines()

def xml_field(phys_line, tag):
    """Pull one tag's value from the <mutation> sitting on this physical XML line."""
    m = re.search(f"<{tag}>(.*?)</{tag}>", xml_lines[phys_line - 1])
    return html.unescape(m.group(1)) if m else None   # &quot; -> "

ok = wrong = 0
bad = []  # (index_no, xml_line, meta_desc, xml_desc, meta_file, xml_file)
for row in meta:
    xl = int(row["xml_line"])
    x_file = xml_field(xl, "sourceFile")
    x_desc = xml_field(xl, "description")
    # meta.source_file is a path (org/.../X.java); the XML stores just the basename
    same_file = x_file == os.path.basename(row["source_file"])
    same_desc = x_desc == row["description"]
    if same_file and same_desc:
        ok += 1
    else:
        wrong += 1
        bad.append((row["index_no"], xl, row["description"], x_desc,
                    os.path.basename(row["source_file"]), x_file))

print(f"rows checked            : {len(meta)}")
print(f"xml_line points correct : {ok}")
print(f"xml_line WRONG          : {wrong}")
print("--- sample wrong ---")
for r in bad[:10]:
    print(f"   idx {r[0]} xml_line {r[1]}: meta {r[2]!r} vs xml {r[3]!r}")

# save any misaligned rows (project name in the filename)
out_path = REPO / "evaluation" / "evaluation_results" / f"{PROJECT}_results" / f"eval0_xml_misalign_{PROJECT}.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)   # create {PROJECT}_results if missing
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["index_no", "xml_line", "meta_description", "xml_description",
                "meta_source_file", "xml_source_file"])
    w.writerows(bad)
print(f"\nsaved {len(bad)} misaligned rows -> {out_path}")


rows checked            : 14019
xml_line points correct : 14019
xml_line WRONG          : 0
--- sample wrong ---

saved 0 misaligned rows -> /Users/nulfat/Documents/Projects/PhD/PITMuS/PITMuS/evaluation/evaluation_results/commons-jexl3_results/eval0_xml_misalign_commons-jexl3.csv


## 1. Count oracle
Did every mutation in the XML become a dataset row?
A gap means some mutants were silently dropped (skipped).

In [22]:
import csv, re, html as _html

n_xml = sum(1 for _ in ET.parse(xml_path).getroot().iter("mutation"))
gap = n_xml - len(meta)
print(f"XML mutations : {n_xml}")
print(f"dataset rows  : {len(meta)}")
print(f"missing       : {gap}")
print("=>", "OK - all reconstructed" if gap == 0 else f"{gap} mutation(s) not reconstructed")

# identify WHICH XML mutations never became a dataset row, and save them.
# every reconstructed row carries the physical xml_line of its <mutation>; any
# <mutation> line not referenced by meta was dropped (skipped) during reconstruction.
with open(xml_path, encoding="utf-8", errors="replace") as _f:
    _xlines = _f.readlines()

def _field(i, tag):
    m = re.search(f"<{tag}>(.*?)</{tag}>", _xlines[i - 1])
    return _html.unescape(m.group(1)) if m else None

recon_lines = {int(r["xml_line"]) for r in meta}
missing_rows = []
for i, ln in enumerate(_xlines, start=1):          # 1-based physical line == xml_line
    if not re.search(r"<mutation[ >]", ln):        # skip non-<mutation> lines (and </mutation>)
        continue
    if i in recon_lines:                           # this mutation was reconstructed
        continue
    missing_rows.append((i, _field(i, "sourceFile"), _field(i, "mutatedClass"),
                         _field(i, "mutatedMethod"), _field(i, "lineNumber"),
                         _field(i, "mutator"), _field(i, "description")))

out_path = REPO / "evaluation" / "evaluation_results" / f"{PROJECT}_results" / f"eval1_not_reconstructed_{PROJECT}.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)   # create {PROJECT}_results if missing
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["xml_line", "source_file", "mutated_class", "mutated_method",
                "line_number", "mutator", "description"])
    w.writerows(sorted(missing_rows))
print(f"\nsaved {len(missing_rows)} not-reconstructed mutation(s) -> {out_path}")


XML mutations : 14030
dataset rows  : 14019
missing       : 11
=> 11 mutation(s) not reconstructed

saved 11 not-reconstructed mutation(s) -> /Users/nulfat/Documents/Projects/PhD/PITMuS/PITMuS/evaluation/evaluation_results/commons-jexl3_results/eval1_not_reconstructed_commons-jexl3.csv


## 2. Mutation faithfulness oracle

PIT's XML says *what* mutation it applied (`description`) but not the mutated source — PITMuS
reconstructs that. This oracle asks: **does the reconstruction reflect that mutation, correctly?**

Every row gets exactly one verdict:

- **PASS** – the edit is a genuine change, the mutant is valid Java, and (for operator mutators)
  the operator that changed matches the `description`.
- **FAIL(reason)** – a real defect. The reasons map to the ways reconstruction goes wrong:
  | reason | what went wrong | example failure mode |
  |---|---|---|
  | `no real change` | only a comment was added; the code is identical | reconstruction gave up (e.g. `removed call` with nested parens) |
  | `mutant is not valid Java` / `broke syntax` | the edit produced un-parseable code | operator matched a generic `<`/`>` bracket |
  | `operator contradicts description` | the changed operator isn't the one PIT named | — |
  | `could not isolate a single operator change` | the recorded line edit isn't one clean operator swap | malformed recorded line |
- **UNVERIFIABLE** – can't soundly judge without more info:
  | reason | why can't tell |
  |---|---|
  | `math ++/-- with a binary +/- also on the line` | which additive operator did PIT hit? needs bytecode position |
  <!-- | `original method does not tokenize` | can't read the baseline to compare against | -->
  

**No false alarms by design.** We only FAIL on things we can prove are wrong.

In [23]:
# ---------- what each operator mutator should do ----------
# maps: description -> (old operator, new operator)
NEG   = {"==": "!=", "!=": "==", ">=": "<", "<=": ">", ">": "<=", "<": ">="}  # negated conditional
BND   = {">=": ">", "<=": "<", ">": ">=", "<": "<="}                          # boundary
MATH  = {"addition": "+", "subtraction": "-", "multiplication": "*", "division": "/", "modulus": "%"}
SHIFT = {
    "Replaced Shift Left with Shift Right":          ("<<", ">>"),
    "Replaced Shift Right with Shift Left":          (">>", "<<"),
    "Replaced Unsigned Shift Right with Shift Left": (">>>", "<<"),
    "Replaced XOR with AND":                         ("^", "&"),
    "Replaced bitwise AND with OR":                  ("&", "|"),
    "Replaced bitwise OR with AND":                  ("|", "&"),
}

def mutator_class(desc):
    """Group the mutator so we know how strictly to check it.
    'math'/'shift'/'negate'/'boundary' = single-operator swap; 'other' = everything else
    (removed call/return/conditional, switch default) which we only check loosely."""
    d = desc.strip()
    if re.match(r"Replaced (?:integer|long|float|double) "
                r"(?:addition|subtraction|multiplication|division|modulus) with", d):
        return "math"
    if d in SHIFT:                        return "shift"
    if d == "negated conditional":        return "negate"
    if d == "changed conditional boundary": return "boundary"
    return "other"

def base_op(tok):
    """Normalize an operator so `+=` and `++` both reduce to '+'. Comparison ops untouched."""
    if tok in ("++", "--"):
        return tok[0]
    if len(tok) >= 2 and tok.endswith("=") and tok not in ("==", "!=", "<=", ">="):
        return tok[:-1]
    return tok

def op_matches_description(desc, old, new):
    """True if the (old -> new) operator change is exactly what `description` says."""
    d = desc.strip()
    m = re.match(r"Replaced (?:integer|long|float|double) (\w+) with (\w+)", d)
    if m:
        return (base_op(old), base_op(new)) == (MATH.get(m.group(1)), MATH.get(m.group(2)))
    if d in SHIFT:
        return (base_op(old), base_op(new)) == SHIFT[d]
    if d == "negated conditional":
        return NEG.get(old) == new
    if d == "changed conditional boundary":
        return BND.get(old) == new
    return False

def op_change(old_line, new_line):
    """Given the recorded before/after line, return the single (old_op, new_op) that changed.
    Works on the short recorded line (one operator change), so multi-char operators that the
    tokenizer splits (>> , >>>) rejoin cleanly. Returns None if it isn't one clean change."""
    a, b = line_tokens(old_line), line_tokens(new_line)
    if a is None or b is None:
        return None
    regions = [(a[i1:i2], b[j1:j2])
               for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=a, b=b).get_opcodes()
               if tag != "equal"]
    if len(regions) != 1:
        return None
    removed, added = regions[0]
    return "".join(removed), "".join(added)

def parses(src):
    """Does this method body parse as Java? (wrapped in a dummy class; no javac/classpath)."""
    try:
        javalang.parse.parse("class _W {\n" + src + "\n}")
        return True
    except Exception:
        return False


def verdict(row):
    """One verdict per row: (PASS|FAIL|UNVERIFIABLE, reason)."""
    m = methods.get(row["index_no"])
    if m is None:
        return "UNVERIFIABLE", "missing method row"

    orig, mut = m["original_method"], m["mutated_method"]
    ot, mt = line_tokens(orig), line_tokens(mut)

    if ot is None:                       # can't judge if we can't even read the original
        return "UNVERIFIABLE", "original method does not tokenize"
    if mt is None:                       # mutant won't tokenize -> broken Java
        return "FAIL", "mutant is not valid Java"
    # PIT's EXPERIMENTAL_SWITCH ("Changed switch default to be first case") reorders the
    # switch jump table (see gen_dataset.reconstruct_switch_default). Where the source block
    # is cleanly reconstructible we emit a faithful bytecode-driven whole-block rewrite: a
    # real, parseable change that is judged like any other non-operator mutator below. Where
    # we cannot reconstruct safely (nested switch, fall-through into the first case, no source
    # `default`, or a default body that maps outside the block) gen_dataset leaves a comment
    # no-op, which surfaces here as no real token change -> UNVERIFIABLE (a tool limitation,
    # not a reconstruction FAIL "no real change").
    if row["description"].strip().lower().startswith("changed switch default"):
        if ot == mt:
            return "UNVERIFIABLE", "switch default not source-representable (reconstruction declined)"
        # else: faithful whole-block rewrite -> fall through to the generic checks below.
    if ot == mt:                         # comments differ at most -> no real mutation applied
        return "FAIL", "no real change (comment-only)"
    if parses(orig) and not parses(mut): # edit broke the syntax (e.g. hit a generic <>)
        return "FAIL", "mutant broke syntax (does not parse)"

    # non-operator mutators: a real, parseable change is all we can soundly require
    if mutator_class(row["description"]) == "other":
        return "PASS", "non-operator mutation applied"     

    # operator mutators: the exact operator swap must match the description
    oc = op_change(row["mutation_line"], row["mutated_line"])
    if oc is None:
        return "FAIL", "operator mutator: could not isolate a single operator change"
    old, new = oc

    # A math +/- landing on ++/-- is FAITHFUL for narrow (short/byte/char) and long operands:
    # they have no increment bytecode, so `i++` compiles to IADD/LADD and IADD->ISUB really is
    # `i--`. (Only an *int* local uses IINC, which PIT reports as "Changed increment", never as
    # "integer addition".) So we only can't trust it when a *binary* +/- ALSO sits on the line --
    # then we can't tell which one PIT hit without the bytecode position.
    if mutator_class(row["description"]) == "math" and ("++" in (old, new) or "--" in (old, new)):
        line_toks = line_tokens(row["mutation_line"]) or []
        if any(t in ("+", "-") for t in line_toks):   # a binary +/- competes for the same mutation
            return "UNVERIFIABLE", "math mutator on ++/-- with a binary +/- also on the line (needs position)"
        # else: lone ++/-- -> fall through; op_matches_description maps ++/-- via base_op and PASSes

    if not op_matches_description(row["description"], old, new):
        return "FAIL", f"operator {old}->{new} contradicts description"
    return "PASS", f"operator {old}->{new} matches description"


# ---------- run the oracle over every row ----------
from collections import Counter

verdicts = Counter()
fail_reasons = Counter()
fails = []  # (index_no, reason, xml_line, description, mutation_line, mutated_line)

for row in meta:
    v, reason = verdict(row)
    verdicts[v] += 1
    if v != "PASS":
        fail_reasons[reason.split("(")[0].strip()] += 1   # group by reason, drop the "(details)"
        fails.append((row["index_no"], f"{v}: {reason}", row.get("xml_line", ""),
                      row["description"], row["mutation_line"], row["mutated_line"]))

print(f"rows checked : {len(meta)}")
for v in ("PASS", "FAIL", "UNVERIFIABLE"):
    print(f"  {v:12} : {verdicts[v]}")
print("--- FAIL / UNVERIFIABLE reasons ---")
for reason, n in fail_reasons.most_common():
    print(f"  {n:5}  {reason}")

# save every non-PASS row so you can inspect them (project name in the filename)
out_path = REPO / "evaluation" / "evaluation_results" / f"{PROJECT}_results" / f"eval2_faithfulness_{PROJECT}.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)   # create {PROJECT}_results if missing
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["index_no", "verdict", "xml_line", "description", "mutation_line", "mutated_line"])
    w.writerows(fails)
print(f"\nsaved {len(fails)} non-PASS rows -> {out_path}")


rows checked : 14019
  PASS         : 13907
  FAIL         : 6
  UNVERIFIABLE : 106
--- FAIL / UNVERIFIABLE reasons ---
    106  switch default not source-representable
      6  operator ~-> contradicts description

saved 112 non-PASS rows -> /Users/nulfat/Documents/Projects/PhD/PITMuS/PITMuS/evaluation/evaluation_results/commons-jexl3_results/eval2_faithfulness_commons-jexl3.csv


## 3. Bytecode ground-truth oracle

Compile each reconstructed mutant and compare its bytecode against the mutant `.class` PIT itself exported (`target/pit-reports/export`) -- PIT's bytecode is the ground truth. Verdicts: **MATCH** / **EQUIVALENT** (confirmed faithful), **BROKEN** (a genuine reconstruction fault that won't compile), **UNREPRESENTABLE** (a faithful mutant that Java *source* can't legally express -- e.g. a constant-`false` loop guard, or removing the only checked-throwing call in a `try`), and **DIVERGENT** (compiles but bytecode differs -- review).

Runs through an in-JVM batch compiler (`PitmusCompile.java`) so `javac` startup is paid once instead of once per mutant.

In [18]:
# ============================================================================
# Section 3: Bytecode ground-truth oracle
#
# For every dataset row we splice the reconstructed `mutated_method` back into
# its real source class, compile it against the project's own compiled classes,
# and compare the resulting bytecode -- method for method -- against the mutant
# .class that PIT itself exported (target/pit-reports/export). PIT's bytecode is
# the ground truth. Verdicts:
#
#   MATCH          our bytecode == PIT's mutant                -> identical
#   EQUIVALENT     our bytecode is a subsequence of PIT's      -> javac const-folded /
#                                                                  dead-code eliminated the
#                                                                  dead branch PIT keeps
#                                                                  (removed-cond / -call /
#                                                                  replaced-return). Same mutant.
#   BROKEN         reconstruction does NOT compile -- and the   -> a genuine reconstruction
#                  error is a real type / syntax / wrong-          fault (wrong operator
#                  occurrence fault                                occurrence, dropped side
#                                                                  effect, ...)
#   UNREPRESENTABLE reconstruction does NOT compile, but only    -> faithful mutant that Java
#                  because Java SOURCE rules forbid what PIT         source simply cannot express
#                  legally does in bytecode (a constant-false       (loop-guard removal ->
#                  loop guard -> "unreachable statement"; the       `for(;false;)`; removing the
#                  removed call was the only checked-throwing        only throwing call in a try).
#                  statement in a try -> "exception ... is never     NOT a PITMuS bug.
#                  thrown"). The edit is correct; source can't hold it.
#   DIVERGENT      compiles but bytecode differs and is not a    -> review (often the same
#                  subsequence                                       dead-code-encoding
#                                                                    difference as EQUIVALENT,
#                                                                    just not a clean subsequence)
#
# Speed: instead of spawning a fresh `javac` JVM per row (~1s each, hours total),
# it compiles in batches through ONE long-lived in-JVM compile server
# (PitmusCompile) that pays javac startup once, and we disassemble our class and
# PIT's in a single combined `javap` call. Set BC_SAMPLE=None for the full run.
# ============================================================================


import subprocess, tempfile, shutil, glob, os, html
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

EXPORT  = proj_dir / "target" / "pit-reports" / "export"
CLASSES = proj_dir / "target" / "classes"
SRCROOT = proj_dir / "src" / "main" / "java"

# --- classpath for compiling reconstructed sources ---
# Base is the project's own compiled classes. Some projects reference external
# Maven dependencies (e.g. joda-convert, commons-logging); without those jars
# javac reports "package ... does not exist" and a CORRECT reconstruction is
# misclassified as BROKEN (false positive). We cache the dependency classpath in
# target/pitmus-deps.cp; if it is missing and the project has a pom.xml, build it
# once via `mvn dependency:build-classpath`. Graceful: no pom / no mvn / mvn
# failure -> fall back to just the project's own compiled classes.
_dep_cp_file = proj_dir / "target" / "pitmus-deps.cp"
if not _dep_cp_file.is_file() and (proj_dir / "pom.xml").is_file() and shutil.which("mvn"):
    print("pitmus-deps.cp missing -> mvn dependency:build-classpath ...")
    _mvn = subprocess.run(
        ["mvn", "-q", "dependency:build-classpath",
         f"-Dmdep.outputFile={_dep_cp_file}"],
        cwd=str(proj_dir), capture_output=True, text=True)
    if _mvn.returncode != 0 or not _dep_cp_file.is_file():
        _err = (_mvn.stderr or _mvn.stdout).strip().splitlines()
        print("  mvn build-classpath failed; deps NOT added ->",
              _err[-1][:120] if _err else "(no output)")

BC_CLASSPATH = str(CLASSES)
if _dep_cp_file.is_file():
    _dep_cp = _dep_cp_file.read_text().strip()
    if _dep_cp:
        BC_CLASSPATH = str(CLASSES) + os.pathsep + _dep_cp
_n_cp = BC_CLASSPATH.count(os.pathsep) + 1
print(f"compile classpath: {_n_cp} entr{'y' if _n_cp == 1 else 'ies'}"
      + ("  (WARNING: no external deps -- Maven dep jars may be missing)" if _n_cp == 1 else ""))

BC_SAMPLE  = None                 # None = check ALL rows; or an int for a quick sample
BC_WORKERS = os.cpu_count() or 8  # process-bound -> scale with cores
BC_CHUNK   = 2000                 # rows per compile-server batch (bounds disk + memory)

# error signatures that mean "faithful mutant, but illegal Java source" (not a fault)
UNREPRESENTABLE_SIGS = ("unreachable statement", "is never thrown")

meta_by_idx = {r["index_no"]: r for r in meta}

def _xf(phys, tag):
    m = re.search(f"<{tag}>(.*?)</{tag}>", xml_lines[phys - 1])
    return html.unescape(m.group(1)) if m else None

def _xindexes(phys):
    seg = re.search(r"<indexes>(.*?)</indexes>", xml_lines[phys - 1])
    return tuple(re.findall(r"<index>(\d+)</index>", seg.group(1))) if seg else ()

# ---- ground truth: map every PIT <mutation> to its exported mutant .class ----
export_index = {}
for det in glob.glob(str(EXPORT / "**" / "details.txt"), recursive=True):
    t = open(det).read()
    def _g(k):
        m = re.search(k + r"=([^,\]]+)", t)
        return m.group(1).strip() if m else None
    mm = re.search(r"mutator=([\w.]+)", t)
    im = re.search(r"indexes=\[([\d, ]*)\]", t)
    key = (_g("clazz"), _g("method"), _g("methodDesc"), _g("lineNumber"),
           mm.group(1) if mm else None,
           tuple(x.strip() for x in im.group(1).split(",") if x.strip()) if im else ())
    cls = glob.glob(os.path.join(os.path.dirname(det), "*.class"))
    export_index[key] = cls[0] if cls else None
print(f"PIT exported mutants indexed: {len(export_index)}")

# ---- build the in-JVM compile server once ----
SERVER_SRC = REPO / "evaluation" / "PitmusCompile.java"
server_dir = Path(tempfile.mkdtemp(prefix="pitmus_srv_"))
_cr = subprocess.run(["javac", "-d", str(server_dir), str(SERVER_SRC)],
                     capture_output=True, text=True)
USE_SERVER = _cr.returncode == 0
print("compile server:", "ready" if USE_SERVER else f"unavailable ({_cr.stderr.strip()[:80]}) -> per-row javac fallback")


def _all_methods_text(out, simple_class):
    """Parse `javap -c -p -s` text; return {(name, descriptor): [normalized instrs]}."""
    res = {}
    cur_name = cur_desc = None
    insns = []; in_code = False
    def flush():
        if cur_name is not None and cur_desc is not None and in_code:
            res[(cur_name, cur_desc)] = list(insns)
    for ln in out.split("\n"):
        s = ln.strip()
        if ("(" in ln and s.endswith(";") and "descriptor:" not in ln
                and not s.startswith("//") and not re.match(r"\d+:", s)):
            flush()
            before = ln[:ln.index("(")].strip()
            name = before.split()[-1] if before.split() else before
            base = name.split(".")[-1].split("$")[-1]
            cur_name = "<init>" if base == simple_class else base
            cur_desc = None; insns = []; in_code = False
            continue
        if s.startswith("descriptor:"):
            cur_desc = s.split("descriptor:", 1)[1].strip(); continue
        if s == "Code:":
            in_code = True; insns = []; continue
        if in_code:
            m = re.match(r"\s*\d+:\s*(\S+)(.*)$", ln)
            if m and m.group(1)[0].isalpha():
                mnem = m.group(1)
                mnem = {"ldc_w": "ldc", "ldc2_w": "ldc2", "goto_w": "goto"}.get(mnem, mnem)
                sym = ""; cm = re.search(r"//\s*(.*)$", m.group(2))
                if cm:
                    sym = re.sub(r"#\d+", "#", re.sub(r"\s+", " ", cm.group(1)).strip())
                insns.append(mnem + (" " + sym if sym else ""))
            elif m:
                pass
            elif s in ("LineNumberTable:", "StackMapTable:", "Exception table:") or (s == "" and insns):
                flush(); in_code = False
    flush()
    return res

def _javap(paths):
    try:
        return subprocess.run(["javap", "-c", "-p", "-s"] + list(paths),
                              capture_output=True, text=True, timeout=60).stdout
    except Exception:
        return None

def disasm_pair(our_cls, pit_cls, meth, desc, simple):
    """One combined javap call for both classes; fall back to two calls if the split fails."""
    out = _javap([our_cls, pit_cls])
    if out is not None:
        parts = out.split("Compiled from")     # both sections carry a SourceFile header
        if len(parts) >= 3:
            a = _all_methods_text(parts[1], simple).get((meth, desc))
            b = _all_methods_text(parts[2], simple).get((meth, desc))
            return a, b
    oa = _javap([our_cls]); ob = _javap([pit_cls])
    a = _all_methods_text(oa, simple).get((meth, desc)) if oa else None
    b = _all_methods_text(ob, simple).get((meth, desc)) if ob else None
    return a, b


def classify_compile_fail(msg):
    return "UNREPRESENTABLE" if any(sig in msg for sig in UNREPRESENTABLE_SIGS) else "BROKEN"


# ---- phase 1/2/3, chunked to bound disk + memory ----
rows_to_check = meta if BC_SAMPLE is None else random.sample(meta, min(BC_SAMPLE, len(meta)))
print(f"bytecode oracle: checking {len(rows_to_check)} rows | workers={BC_WORKERS} | chunk={BC_CHUNK}")

bc_verdicts = Counter()
bc_rows = []
_src_cache = {}

def _run_compiles(requests, classpath):
    """Return {idx: (status, msg)} using the server (or per-row javac fallback)."""
    res = {}
    if not requests:
        return res
    if USE_SERVER:
        proc = subprocess.run(["java", "-cp", str(server_dir), "PitmusCompile",
                               str(classpath), str(BC_WORKERS)],
                              input="\n".join(requests), capture_output=True, text=True, timeout=3600)
        for ln in proc.stdout.splitlines():
            parts = ln.split("\t", 2)
            if len(parts) >= 2:
                res[parts[0]] = (parts[1], parts[2] if len(parts) > 2 else "")
        return res
    for req in requests:                                    # fallback: one javac per row
        idx, src, out = req.split("\t")
        r = subprocess.run(["javac", "--release", "8", "-cp", str(classpath), "-d", out, src],
                           capture_output=True, text=True, timeout=180)
        if r.returncode == 0:
            res[idx] = ("OK", "")
        else:
            first = r.stderr.strip().split("\n")[0] if r.stderr.strip() else "compile failed"
            res[idx] = ("ERR", first)
    return res

def _finish(idx, jm, status, msg):
    if status != "OK":
        return idx, classify_compile_fail(msg), msg[:120]
    if jm["pit"] is None:
        return idx, "UNMAPPED", None
    our = os.path.join(jm["out"], *jm["fqn"].split(".")) + ".class"
    if not os.path.exists(our):
        return idx, "NO_OUR_CLASS", None
    a, b = disasm_pair(our, jm["pit"], jm["meth"], jm["desc"], jm["simple"])
    if a is None or b is None:
        return idx, "DISASM_ERR", f"a={'None' if a is None else 'ok'} b={'None' if b is None else 'ok'} {jm['meth']}{jm['desc']}"
    if a == b:
        return idx, "MATCH", None
    it = iter(b)
    if all(tok in it for tok in a):
        return idx, "EQUIVALENT", None
    return idx, "DIVERGENT", f"{jm['meth']}{jm['desc']} | alen={len(a)} blen={len(b)}"

for c0 in range(0, len(rows_to_check), BC_CHUNK):
    chunk = rows_to_check[c0:c0 + BC_CHUNK]
    batch = Path(tempfile.mkdtemp(prefix="pitmus_bc_"))
    requests = []; jobmeta = {}
    try:
        # phase 1: splice + write sources, collect compile requests
        for row in chunk:
            idx = row["index_no"]; m = methods.get(idx)
            if not m:
                bc_verdicts["NOMETHOD"] += 1; bc_rows.append((idx, "NOMETHOD", "", "")); continue
            xl = int(row["xml_line"])
            fqn = _xf(xl, "mutatedClass"); meth = _xf(xl, "mutatedMethod"); desc = _xf(xl, "methodDescription")
            simple = fqn.split(".")[-1].split("$")[-1]
            key = (fqn, meth, desc, _xf(xl, "lineNumber"), _xf(xl, "mutator"), _xindexes(xl))
            pit = export_index.get(key)
            sf = SRCROOT / row["source_file"]
            if not sf.exists():
                bc_verdicts["NOSRC"] += 1; bc_rows.append((idx, "NOSRC", "", "")); continue
            text = _src_cache.get(sf)
            if text is None:
                text = sf.read_text(); _src_cache[sf] = text
            if m["original_method"] not in text:
                bc_verdicts["NOSPLICE"] += 1; bc_rows.append((idx, "NOSPLICE", "", "")); continue
            new_text = text.replace(m["original_method"], m["mutated_method"], 1)
            wd = batch / idx; outd = wd / "out"; outd.mkdir(parents=True)
            sp = wd / sf.name; sp.write_text(new_text)
            requests.append(f"{idx}\t{sp}\t{outd}")
            jobmeta[idx] = dict(fqn=fqn, meth=meth, desc=desc, simple=simple, pit=pit, out=str(outd))

        # phase 2: compile the whole chunk in one JVM
        compile_res = _run_compiles(requests, BC_CLASSPATH)

        # phase 3: classify + disassemble (javap parallel)
        with ThreadPoolExecutor(max_workers=BC_WORKERS) as ex:
            futs = [ex.submit(_finish, idx, jm, *compile_res.get(idx, ("ERR", "no result")))
                    for idx, jm in jobmeta.items()]
            for fut in as_completed(futs):
                idx, verdict, detail = fut.result()
                bc_verdicts[verdict] += 1
                if verdict not in ("MATCH", "EQUIVALENT"):
                    desc = meta_by_idx.get(idx, {}).get("description", "")
                    bc_rows.append((idx, verdict, desc, detail or ""))
    finally:
        shutil.rmtree(batch, ignore_errors=True)
    print(f"  ...{min(c0 + BC_CHUNK, len(rows_to_check))}/{len(rows_to_check)} done")

shutil.rmtree(server_dir, ignore_errors=True)

bc_confirmed = bc_verdicts["MATCH"] + bc_verdicts["EQUIVALENT"]
print("\n=== bytecode ground-truth verdicts ===")
for k, v in bc_verdicts.most_common():
    print(f"  {k:16} {v}")
print(f"\n  confirmed faithful (MATCH+EQUIVALENT)   : {bc_confirmed}/{len(rows_to_check)}")
print(f"  BROKEN (genuine reconstruction faults)  : {bc_verdicts['BROKEN']}")
print(f"  UNREPRESENTABLE (faithful, illegal Java): {bc_verdicts['UNREPRESENTABLE']}")
print(f"  DIVERGENT (needs review)                : {bc_verdicts['DIVERGENT']}")

# split the non-confirmed rows into two CSVs.
#
# "broken" = ONLY genuine reconstruction faults: the reconstruction does NOT
#   compile for a real reason (BROKEN). These are the true reconstruction bugs.
#
# "other"  = every other non-confirmed row -- none of which is a confirmed
#   reconstruction fault:
#     DIVERGENT        compiles but bytecode differs & isn't a clean subsequence.
#                      "needs review" -- in practice mostly the same dead-code
#                      encoding difference as EQUIVALENT (javac drops dead code
#                      PIT keeps), NOT a broken reconstruction.
#     UNREPRESENTABLE  faithful mutant that Java source can't legally express.
#     UNMAPPED         no PIT-exported .class to compare against (can't test).
#     NO_OUR_CLASS / DISASM_ERR / NOMETHOD / NOSRC / NOSPLICE
#                      setup / tooling abstentions -- we couldn't run the check.
broken_rows = [r for r in bc_rows if r[1] == "BROKEN"]
other_rows  = [r for r in bc_rows if r[1] != "BROKEN"]

res_dir = REPO / "evaluation" / "evaluation_results" / f"{PROJECT}_results"
res_dir.mkdir(parents=True, exist_ok=True)

def _write_bc(rows, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        wtr = csv.writer(f)
        wtr.writerow(["index_no", "verdict", "description", "detail"])
        wtr.writerows(sorted(rows, key=lambda r: (r[1], int(r[0]))))

bc_broken_csv = res_dir / f"eval3_bytecode_{PROJECT}_broken.csv"
bc_other_csv  = res_dir / f"eval3_bytecode_{PROJECT}_other.csv"
_write_bc(broken_rows, bc_broken_csv)
_write_bc(other_rows, bc_other_csv)
print(f"\nsaved {len(broken_rows)} broken (genuine BROKEN reconstruction faults) rows -> {bc_broken_csv}")
print(f"saved {len(other_rows)} other (DIVERGENT + UNREPRESENTABLE + setup/abstention) rows -> {bc_other_csv}")


compile classpath: 6 entries
PIT exported mutants indexed: 967
compile server: ready
bytecode oracle: checking 967 rows | workers=10 | chunk=2000
  ...967/967 done

=== bytecode ground-truth verdicts ===
  EQUIVALENT       876
  UNREPRESENTABLE  45
  DIVERGENT        24
  MATCH            22

  confirmed faithful (MATCH+EQUIVALENT)   : 898/967
  BROKEN (genuine reconstruction faults)  : 0
  UNREPRESENTABLE (faithful, illegal Java): 45
  DIVERGENT (needs review)                : 24

saved 0 broken (genuine BROKEN reconstruction faults) rows -> /Users/nulfat/Documents/Projects/PhD/PITMuS/PITMuS/evaluation/evaluation_results/commons-dbutils_results/eval3_bytecode_commons-dbutils_broken.csv
saved 69 other (DIVERGENT + UNREPRESENTABLE + setup/abstention) rows -> /Users/nulfat/Documents/Projects/PhD/PITMuS/PITMuS/evaluation/evaluation_results/commons-dbutils_results/eval3_bytecode_commons-dbutils_other.csv


## 4. Consolidated report

In [19]:
# ============================================================================
# Section 4: write a consolidated Evaluation-<project>.txt with every oracle's result.
# Reads the in-memory result variables produced by Sections 0/1/2/3.
# ============================================================================
import datetime

g = globals()
L = []                                   # report lines
def w(s=""): L.append(s)

w("=" * 70)
w(f" PITMuS reconstruction evaluation  --  project: {PROJECT}")
w(f" dataset  : {DATASET}")
w(f" generated: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
w("=" * 70)
w(f"total reconstructed mutations (dataset rows): {len(meta)}")
w("")

# ---- Section 0: XML alignment oracle -------------------------------------
w("-" * 70)
w("Section 0  XML alignment oracle  (each dataset row points at the right <mutation>)")
w("-" * 70)
if "ok" in g and "wrong" in g:
    w(f"  rows checked            : {len(meta)}")
    w(f"  xml_line points correct : {ok}")
    w(f"  xml_line WRONG          : {wrong}")
    w(f"  => {'OK - all aligned' if wrong == 0 else str(wrong) + ' MISALIGNED (see eval0 CSV)'}")
    w(f"  detail csv: eval0_xml_misalign_{PROJECT}.csv")
else:
    w("  (Section 0 not run)")
w("")

# ---- Section 1: count oracle ---------------------------------------------
w("-" * 70)
w("Section 1  Count oracle  (every XML mutation was reconstructed)")
w("-" * 70)
if "n_xml" in g and "gap" in g:
    w(f"  XML mutations : {n_xml}")
    w(f"  dataset rows  : {len(meta)}")
    w(f"  missing       : {gap}")
    w(f"  => {'OK - all reconstructed' if gap == 0 else str(gap) + ' NOT reconstructed'}")
else:
    w("  (Section 1 not run)")
w("")

# ---- Section 2: faithfulness oracle --------------------------------------
w("-" * 70)
w("Section 2  Faithfulness oracle  (the reconstructed edit matches the mutator)")
w("-" * 70)
if "verdicts" in g:
    w(f"  rows checked : {sum(verdicts.values())}")
    for v in ("PASS", "FAIL", "UNVERIFIABLE"):
        w(f"    {v:12} : {verdicts[v]}")
    if "fail_reasons" in g and fail_reasons:
        w("  FAIL / UNVERIFIABLE reasons:")
        for reason, n in fail_reasons.most_common():
            w(f"    {n:5}  {reason}")
    w(f"  detail csv: eval2_faithfulness_{PROJECT}.csv")
else:
    w("  (Section 2 not run)")
w("")

# ---- Section 3: bytecode ground-truth oracle -----------------------------
w("-" * 70)
w("Section 3  Bytecode ground-truth oracle  (recon compiles + matches PIT's exported mutant .class)")
w("-" * 70)
if "bc_verdicts" in g:
    checked = sum(bc_verdicts.values())
    confirmed = bc_verdicts["MATCH"] + bc_verdicts["EQUIVALENT"]
    w(f"  rows checked                          : {checked}")
    w(f"    MATCH      (bytecode identical)     : {bc_verdicts['MATCH']}")
    w(f"    EQUIVALENT (subsequence of PIT)     : {bc_verdicts['EQUIVALENT']}")
    w(f"    => confirmed faithful               : {confirmed}")
    w(f"    BROKEN     (real fault, won't compile): {bc_verdicts['BROKEN']}   <- genuine reconstruction faults")
    w(f"    UNREPRESENTABLE (faithful, illegal Java): {bc_verdicts['UNREPRESENTABLE']}   <- mutant PIT makes in bytecode but source can't express")
    w(f"    DIVERGENT  (bytecode differs)       : {bc_verdicts['DIVERGENT']}   <- needs review")
    setup = {k: bc_verdicts[k] for k in
             ("UNMAPPED", "NO_OUR_CLASS", "DISASM_ERR", "NOMETHOD", "NOSRC", "NOSPLICE")
             if bc_verdicts.get(k)}
    if setup:
        w("    setup / unverifiable (abstained):")
        for k, n in setup.items():
            w(f"      {k:14} {n}")
    w(f"  detail csv (genuine faults): eval3_bytecode_{PROJECT}_broken.csv")
    w(f"  detail csv (other non-faults): eval3_bytecode_{PROJECT}_other.csv")
else:
    w("  (Section 3 not run)")
w("")

# ---- overall roll-up ------------------------------------------------------
w("=" * 70)
w(" OVERALL")
w("=" * 70)
w(f"  faithfulness PASS rate         : {verdicts['PASS']}/{sum(verdicts.values())}" if "verdicts" in g else "  faithfulness: n/a")
if "bc_verdicts" in g:
    checked = sum(bc_verdicts.values())
    confirmed = bc_verdicts["MATCH"] + bc_verdicts["EQUIVALENT"]
    w(f"  bytecode confirmed faithful      : {confirmed}/{checked}")
    w(f"  bytecode BROKEN (real faults)    : {bc_verdicts['BROKEN']}   <- genuine reconstruction faults")
    w(f"  bytecode UNREPRESENTABLE         : {bc_verdicts['UNREPRESENTABLE']}   <- faithful, but illegal Java source (not a fault)")
    w(f"  bytecode DIVERGENT (review)      : {bc_verdicts['DIVERGENT']}")
else:
    w("  bytecode oracle: n/a")
w("=" * 70)

report = "\n".join(L) + "\n"
out_txt = REPO / "evaluation" / "evaluation_results" / f"{PROJECT}_results" / f"Evaluation-{PROJECT}_{DATASET_VERSION}.txt"
out_txt.write_text(report, encoding="utf-8")
print(report)
print(f"saved report -> {out_txt}")


 PITMuS reconstruction evaluation  --  project: commons-dbutils
 dataset  : PITMuS_dataset
 generated: 2026-08-06 12:06:11
total reconstructed mutations (dataset rows): 967

----------------------------------------------------------------------
Section 0  XML alignment oracle  (each dataset row points at the right <mutation>)
----------------------------------------------------------------------
  rows checked            : 967
  xml_line points correct : 967
  xml_line WRONG          : 0
  => OK - all aligned
  detail csv: eval0_xml_misalign_commons-dbutils.csv

----------------------------------------------------------------------
Section 1  Count oracle  (every XML mutation was reconstructed)
----------------------------------------------------------------------
  XML mutations : 967
  dataset rows  : 967
  missing       : 0
  => OK - all reconstructed

----------------------------------------------------------------------
Section 2  Faithfulness oracle  (the reconstructed edit match